# Chapter 12: Fine-Tuning Generation Models with LoRA

> **Goal:** Apply parameter-efficient fine-tuning to customize LLMs cheaply.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained('distilgpt2')
tokenizer = AutoTokenizer.from_pretrained('distilgpt2')
tokenizer.pad_token = tokenizer.eos_token

original = sum(p.numel() for p in model.parameters())
print(f'Original parameters: {original:,}')

lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05, target_modules=['c_attn'])
model = get_peft_model(model, lora_cfg)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
pct = 100 * trainable / (trainable + frozen)

print(f'LoRA trainable:  {trainable:,} ({pct:.2f}%)')
print(f'Frozen (base):   {frozen:,}')
print(f'Memory savings:  ~{original//trainable}x')
print('\nLoRA adds tiny adapter matrices to attention layers.')
print('Only the adapters are trained - the base model stays frozen.')
print('After training, adapters can be merged: zero inference overhead!')
model.print_trainable_parameters()

---

## Summary

Return to [README](../../README.md).